In [ ]:
import subprocess
import os
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import re

# Configuration
img_size = 800
batch_size = 16
epochs = 50
dataset_yaml = os.path.abspath("dataset2.yaml")
weights = os.path.abspath("yolov5.pt")
model_name = "sgsl_model_improved v3"

print("🚀 Starting YOLOv5 training...\n")

# Run YOLOv5 training with live log processing
try:
    process = subprocess.Popen(
        [
            "python", "yolov5/train.py",
            "--img", str(img_size),
            "--batch", str(batch_size),
            "--epochs", str(epochs),
            "--data", dataset_yaml,
            "--weights", weights,
            "--name", model_name,
            "--cache"
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    current_epoch = None
    for line in process.stdout:
        line = line.strip()

        # Clean YOLO epoch log (matches e.g. "1/29 ...")
        epoch_match = re.match(r'^(\d+)/(\d+)\s+(.*)', line)
        if epoch_match:
            epoch_num = epoch_match.group(1)
            epoch_total = epoch_match.group(2)
            epoch_data = epoch_match.group(3)
            print(f"📘 Epoch {epoch_num}/{epoch_total} — {epoch_data}")
        elif 'all' in line and 'mAP@' in line:
            # Prints metrics like Precision / Recall / mAP
            print(f"🔍 {line}")
        elif 'train/box_loss' in line or 'val/box_loss' in line:
            print(f"📊 {line}")
        else:
            # Optional: uncomment below if you want to see all raw logs too
            # print(line)
            pass

    process.wait()
    if process.returncode != 0:
        raise subprocess.CalledProcessError(process.returncode, process.args)

except subprocess.CalledProcessError as e:
    print("❌ Training failed!")
    print(e)
    exit(1)

run_folder = f"runs/train/{model_name}"
print(f"\n✅ Training complete. Check the folder: {run_folder}/")